# Bike-Share Historical Analysis

Exploratory analysis of Berlin Nextbike station availability data.
Covers four analytical themes:
1. Station variability (AVG vs STD scatter)
2. Geographic risk distribution
3. Temporal patterns (hourly, daily, weekly heatmap, 7-day MA)
4. Weather impact on availability

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

load_dotenv()

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})

PALETTE = sns.color_palette('Blues_r', 6)

def get_engine():
    host = os.getenv('MYSQL_HOST', 'localhost')
    port = os.getenv('MYSQL_PORT', '3306')
    user = os.getenv('MYSQL_USER', 'root')
    pw   = os.getenv('MYSQL_PASSWORD', '')
    db   = os.getenv('MYSQL_DB', 'bikeshare')
    return create_engine(f'mysql+pymysql://{user}:{pw}@{host}:{port}/{db}?charset=utf8mb4',
                         pool_pre_ping=True)

engine = get_engine()
print('Connected to MySQL ✓')

## 1 · Load Data

In [ ]:
# Station-level aggregates (from station_hourly)
station_agg = pd.read_sql("""
    SELECT
        station_id,
        AVG(avg_availability_pct)   AS avg_avail,
        STD(avg_availability_pct)   AS std_avail,
        AVG(avg_free_bikes)         AS avg_free_bikes,
        SUM(empty_count)            AS total_empty_intervals,
        COUNT(*)                    AS n_hours
    FROM station_hourly
    GROUP BY station_id
    HAVING n_hours >= 24
""", engine)

print(f'Stations: {len(station_agg):,}')
station_agg.describe().round(2)

In [ ]:
# Hourly time series (sampled for memory)
hourly_ts = pd.read_sql("""
    SELECT hour_ts, AVG(avg_availability_pct) AS system_avail,
           SUM(CASE WHEN avg_availability_pct < 10 THEN 1 ELSE 0 END) AS empty_stations,
           COUNT(DISTINCT station_id) AS n_stations
    FROM station_hourly
    GROUP BY hour_ts
    ORDER BY hour_ts
""", engine, parse_dates=['hour_ts'])

print(f'Date range: {hourly_ts.hour_ts.min()} → {hourly_ts.hour_ts.max()}')
print(f'Hours: {len(hourly_ts):,}')

In [ ]:
# Weather + availability joined
weather_join = pd.read_sql("""
    SELECT
        h.hour_ts,
        AVG(h.avg_availability_pct) AS system_avail,
        w.temperature_c,
        w.humidity_pct,
        w.is_rain,
        w.is_snow
    FROM station_hourly h
    JOIN weather_observations w
        ON DATE_FORMAT(w.observed_ts, '%Y-%m-%d %H:00:00') = h.hour_ts
    GROUP BY h.hour_ts, w.temperature_c, w.humidity_pct, w.is_rain, w.is_snow
""", engine, parse_dates=['hour_ts'])

print(f'Weather rows: {len(weather_join):,}')

## 2 · Station Variability Scatter (AVG vs STD)

In [ ]:
def risk_label(row):
    if row.avg_avail < 15:
        return 'Persistently Empty'
    elif row.avg_avail > 80:
        return 'Persistently Full'
    elif row.std_avail > 25:
        return 'Volatile'
    else:
        return 'Stable'

station_agg['risk'] = station_agg.apply(risk_label, axis=1)

palette = {
    'Persistently Empty': '#e74c3c',
    'Persistently Full':  '#3498db',
    'Volatile':           '#f39c12',
    'Stable':             '#2ecc71',
}

fig, ax = plt.subplots(figsize=(10, 6))
for label, grp in station_agg.groupby('risk'):
    ax.scatter(grp['avg_avail'], grp['std_avail'],
               label=f"{label} (n={len(grp)})",
               color=palette[label], alpha=0.6, s=30, edgecolors='none')

ax.axvline(15, color='red',   ls='--', lw=1, label='Empty threshold (15%)')
ax.axvline(80, color='blue',  ls='--', lw=1, label='Full threshold (80%)')
ax.axhline(25, color='orange',ls='--', lw=1, label='Volatility threshold (σ=25)')

ax.set_xlabel('Average Availability (%)')
ax.set_ylabel('Std Dev of Availability (%)')
ax.set_title('Station Variability — AVG vs STD', fontsize=14, fontweight='bold')
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.savefig('../data/station_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

print(station_agg['risk'].value_counts())

## 3 · Temporal Patterns

In [ ]:
# 3a — Hourly system availability + 7-day rolling average
ts = hourly_ts.set_index('hour_ts')['system_avail']
rolling_7d = ts.rolling(window=168, min_periods=24).mean()

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(ts.index, ts.values, alpha=0.3, color='#3498db', lw=0.8, label='Hourly avg')
ax.plot(rolling_7d.index, rolling_7d.values, color='#2c3e50', lw=2, label='7-day MA')
ax.set_ylabel('System Availability (%)')
ax.set_title('System-Wide Availability Over Time', fontsize=13, fontweight='bold')
ax.legend()
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=100))
plt.tight_layout()
plt.savefig('../data/temporal_trend.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 3b — Hour × Day-of-Week heatmap
hourly_ts['hour']  = hourly_ts['hour_ts'].dt.hour
hourly_ts['dow']   = hourly_ts['hour_ts'].dt.dayofweek

pivot = hourly_ts.groupby(['hour', 'dow'])['system_avail'].mean().unstack()
pivot.columns = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(pivot, cmap='Blues', annot=True, fmt='.0f', linewidths=0.4,
            cbar_kws={'label': 'Mean Availability (%)'}, ax=ax)
ax.set_xlabel('')
ax.set_ylabel('Hour of Day')
ax.set_title('Availability Heatmap — Hour × Day of Week', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/heatmap_hour_dow.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 3c — Average hourly profile (weekday vs weekend)
hourly_ts['is_weekend'] = (hourly_ts['dow'] >= 5).astype(int)

profile = hourly_ts.groupby(['hour', 'is_weekend'])['system_avail'].mean().unstack()
profile.columns = ['Weekday', 'Weekend']

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(profile.index, profile['Weekday'], marker='o', label='Weekday', color='#2c3e50')
ax.plot(profile.index, profile['Weekend'], marker='s', label='Weekend', color='#e67e22')

for h in [7, 8, 9, 17, 18, 19]:
    ax.axvspan(h - 0.5, h + 0.5, alpha=0.07, color='red')

ax.set_xticks(range(24))
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Mean Availability (%)')
ax.set_title('Hourly Availability Profile — Weekday vs Weekend', fontsize=13, fontweight='bold')
ax.legend()
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=100))
plt.tight_layout()
plt.savefig('../data/hourly_profile.png', dpi=150, bbox_inches='tight')
plt.show()

## 4 · Weather Impact Analysis

In [ ]:
# 4a — Temperature vs availability scatter
df_w = weather_join.dropna(subset=['temperature_c', 'system_avail'])

fig, ax = plt.subplots(figsize=(9, 5))
sc = ax.scatter(df_w['temperature_c'], df_w['system_avail'],
                c=df_w['humidity_pct'], cmap='coolwarm_r',
                alpha=0.4, s=20, edgecolors='none')
plt.colorbar(sc, ax=ax, label='Humidity (%)')

# Trend line
z = np.polyfit(df_w['temperature_c'], df_w['system_avail'], 1)
p = np.poly1d(z)
xs = np.linspace(df_w['temperature_c'].min(), df_w['temperature_c'].max(), 100)
ax.plot(xs, p(xs), color='black', lw=2, label=f'Trend (slope={z[0]:.2f})')

ax.set_xlabel('Temperature (°C)')
ax.set_ylabel('System Availability (%)')
ax.set_title('Temperature vs Availability', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('../data/weather_temp_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 4b — Rain impact: box plot
df_w['Condition'] = df_w['is_rain'].map({0: 'Dry', 1: 'Rainy'})

fig, ax = plt.subplots(figsize=(7, 5))
sns.boxplot(data=df_w, x='Condition', y='system_avail',
            palette={'Dry': '#3498db', 'Rainy': '#7f8c8d'}, ax=ax)

dry_med   = df_w[df_w['is_rain'] == 0]['system_avail'].median()
rainy_med = df_w[df_w['is_rain'] == 1]['system_avail'].median()

ax.set_ylabel('System Availability (%)')
ax.set_title('Availability: Dry vs Rainy Conditions', fontsize=13, fontweight='bold')
ax.text(0, dry_med + 1, f'Median: {dry_med:.1f}%', ha='center', fontsize=10)
ax.text(1, rainy_med + 1, f'Median: {rainy_med:.1f}%', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('../data/rain_impact.png', dpi=150, bbox_inches='tight')
plt.show()

from scipy import stats
dry   = df_w[df_w['is_rain'] == 0]['system_avail']
rainy = df_w[df_w['is_rain'] == 1]['system_avail']
t_stat, p_val = stats.mannwhitneyu(dry, rainy, alternative='two-sided')
print(f'Mann-Whitney U: stat={t_stat:.1f}, p={p_val:.4f}')
print(f'Dry median: {dry.median():.2f}%  |  Rainy median: {rainy.median():.2f}%')
print(f'Difference: {dry.median() - rainy.median():.2f} pp')

## 5 · Summary Statistics

In [ ]:
print('=== Dataset Overview ===')
print(f"Stations analysed:     {len(station_agg):,}")
print(f"Date range:            {hourly_ts.hour_ts.min()} → {hourly_ts.hour_ts.max()}")
print()
print('=== System-Wide Availability ===')
print(f"Mean:   {station_agg.avg_avail.mean():.1f}%")
print(f"Median: {station_agg.avg_avail.median():.1f}%")
print()
print('=== Station Risk Distribution ===')
print(station_agg['risk'].value_counts())
print()
print('=== Peak vs Off-Peak ===')
peak_hours    = hourly_ts[hourly_ts['hour'].isin([7,8,9,17,18,19])]['system_avail']
offpeak_hours = hourly_ts[~hourly_ts['hour'].isin([7,8,9,17,18,19])]['system_avail']
print(f"Peak hours:    {peak_hours.mean():.1f}% avg availability")
print(f"Off-peak hours:{offpeak_hours.mean():.1f}% avg availability")